# LAI Feature Merge Pipeline

**Purpose:** Join the daily Leaf Area Index (LAI) dataset (1994–2021) onto the
weather + population feature table, adding `LAI` as a new column keyed by
`(lon, lat, day)`.

**Pipeline Overview:**

| Phase | Steps |
|-------|-------|
| **A. Load & Inspect LAI** | Load LAI parquet; verify shape, dtypes, date range, and key uniqueness |
| **B. Load & Inspect Base** | Load weather+population feature table; confirm date range alignment with LAI |
| **C. Merge & Validate** | Left-join on `(lon, lat, day)`; assert row count unchanged; check missing rates; save output |

**Inputs:**
- `Clean_Data/LAI_Data/LAI.parquet` — daily LAI, veg-filtered
  *(from `01_02_Data_Clean_-_LAI.ipynb`)*
- `Clean_Data/Feature_Data/Weather_Population_Merged.parquet` — weather + population feature table
  *(from `03_02_Merge_Features_-_Population_Data.ipynb`)*

**Output:**
- `Clean_Data/Feature_Data/Weather_Population_LAI_Merged.parquet` — weather + population + LAI (127M rows × 18 cols)

> **Join strategy:** `LEFT` join on `(lon, lat, day)` — every weather row is kept.
> LAI extends to 2021-12-27 while weather ends at 2020-09-30, so all weather rows
> will find a matching LAI row. Small `NaN` fraction (~3%) is expected from
> cloud-contaminated LAI retrievals.

## 0. Configuration

Centralized path configuration — **edit this cell only** to adapt to your local setup.

In [5]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input: veg-filtered LAI parquet (from 01_02_Data_Clean_-_LAI.ipynb)
LAI_PATH = os.path.join(
    PROJECT_ROOT, "Clean_Data", "Extended_Data_w_Veg_Filter", "Vegetation", "LAI.parquet"
)

# Input: weather + population feature table (from 03_02_Merge_Features_-_Population_Data.ipynb)
WEATHER_POP_PATH = os.path.join(
    PROJECT_ROOT, "Clean_Data", "Feature_Data",
    "Weather_Population_Merged.parquet"
)

# Output
FEATURE_DATA_DIR = os.path.join(PROJECT_ROOT, "Clean_Data", "Feature_Data")
OUTPUT_FILE      = "Weather_Population_LAI_Merged.parquet"

os.makedirs(FEATURE_DATA_DIR, exist_ok=True)

print(f"LAI path          : {LAI_PATH}")
print(f"Weather+pop path  : {WEATHER_POP_PATH}")
print(f"Output dir        : {FEATURE_DATA_DIR}")

# Quick existence checks
for label, path in [("LAI_PATH",         LAI_PATH),
                    ("WEATHER_POP_PATH", WEATHER_POP_PATH)]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {label}")

LAI path          : E:\zcao\CA_Wildfire\Clean_Data\Extended_Data_w_Veg_Filter\Vegetation\LAI.parquet
Weather+pop path  : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Weather_Population_Merged.parquet
Output dir        : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data
  [OK] LAI_PATH
  [OK] WEATHER_POP_PATH


## 1. Environment Setup

In [6]:
import sys
import gc
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pyproj
from datetime import datetime

gc.collect()

print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print(f"pyproj : {pyproj.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2
numpy  : 1.24.4
pyproj : 3.6.1


---

# Phase A: Load & Inspect LAI Data

Load the veg-filtered LAI parquet and run QA checks before the join:
shape, dtypes, date range, missing rate, and key uniqueness.
The key uniqueness check (`time × lat × lon`) is critical — any duplicates
would cause row multiplication in the left join.

## 2. Load LAI Data

LAI is a daily gridded product covering 1994–2021. The `time` column will be
renamed to `day` before the join to match the weather feature table convention.

In [7]:
lai_dat = pd.read_parquet(LAI_PATH)

print(f"Shape   : {lai_dat.shape}")
print(f"Columns : {list(lai_dat.columns)}")
print(f"dtypes  :\n{lai_dat.dtypes.to_string()}")

Shape   : (147065536, 4)
Columns : ['time', 'lat', 'lon', 'LAI']
dtypes  :
time    datetime64[ns]
lat            float64
lon            float64
LAI            float64


In [8]:
print(f"Date range  : {lai_dat['time'].min().date()} → {lai_dat['time'].max().date()}")
print()
missing_lai = lai_dat.isnull().mean().mul(100).rename('missing_%')
print("Missing rate (%):\n", missing_lai.to_string())

Date range  : 1994-01-01 → 2021-12-27

Missing rate (%):
 time    0.000000
lat     0.000000
lon     0.000000
LAI     5.594224


## 3. Key Uniqueness Check

Assert that every `(time, lat, lon)` triplet is unique in the LAI dataset.
Duplicates would silently multiply rows in the downstream left join.

In [9]:
n_rows   = lai_dat.shape[0]
n_unique = lai_dat[['time', 'lat', 'lon']].drop_duplicates().shape[0]

print(f"Total rows          : {n_rows:,}")
print(f"Unique (time,lat,lon): {n_unique:,}")

assert n_unique == n_rows, \
    f"Duplicate keys found: {n_rows - n_unique:,} duplicates"
print("Key uniqueness OK — no duplicates.")

Total rows          : 147,065,536
Unique (time,lat,lon): 147,065,536
Key uniqueness OK — no duplicates.


---

# Phase B: Load & Inspect Base Feature Table

Load the weather + population feature table and verify its date range
overlaps fully with the LAI data. Since LAI extends beyond the weather
end date (2021 vs 2020), all weather rows should find a match.

## 4. Rename LAI `time` → `day`

Align the LAI column name with the weather feature table convention
before loading the large base table.

In [10]:
lai_dat = lai_dat.rename(columns={'time': 'day'})

print(f"LAI date range : {lai_dat['day'].min().date()} → {lai_dat['day'].max().date()}")

LAI date range : 1994-01-01 → 2021-12-27


## 5. Load Weather + Population Feature Table

Load the wide feature table produced by `03_02_Merge_Features_-_Population_Data.ipynb`.
Confirm shape, dtypes, and that its date range is fully covered by the LAI data.

In [11]:
all_features = pd.read_parquet(WEATHER_POP_PATH)

print(f"Shape   : {all_features.shape}")
print(f"\nColumn dtypes:")
print(all_features.dtypes.to_string())

Shape   : (127478960, 17)

Column dtypes:
day                                          datetime64[ns]
lat                                                 float64
lon                                                 float64
SWE                                                 float32
year                                                  int32
dead_fuel_moisture_1000hr                           float64
dead_fuel_moisture_100hr                            float64
max_air_temperature                                 float64
max_relative_humidity                               float64
min_air_temperature                                 float64
min_relative_humidity                               float64
precipitation_amount                                float64
specific_humidity                                   float64
surface_downwelling_shortwave_flux_in_air           float64
wind_from_direction                                 float32
wind_speed                                          float6

In [12]:
feat_min = all_features['day'].min()
feat_max = all_features['day'].max()
lai_min  = lai_dat['day'].min()
lai_max  = lai_dat['day'].max()

print(f"Weather date range : {feat_min.date()} → {feat_max.date()}")
print(f"LAI date range     : {lai_min.date()} → {lai_max.date()}")

# Weather range must be fully contained within LAI range
if feat_min >= lai_min and feat_max <= lai_max:
    print("Date coverage OK — weather range fully within LAI range.")
else:
    print("WARNING — weather dates outside LAI coverage; some rows will get NaN LAI.")

Weather date range : 1994-01-01 → 2020-09-30
LAI date range     : 1994-01-01 → 2021-12-27
Date coverage OK — weather range fully within LAI range.


---

# Phase C: Merge & Validate

Left-join the LAI data onto the weather + population feature table on
`(lon, lat, day)`. A left join ensures no weather rows are dropped.
The row count must be identical before and after.

## 6. Left-Join LAI onto Feature Table

Join key: `(lon, lat, day)`. Each grid cell × day combination in the weather
table picks up the corresponding daily LAI value.

In [13]:
rows_before = len(all_features)
print(f"Before merge : {rows_before:,} rows × {all_features.shape[1]} cols")

all_features = pd.merge(
    all_features, lai_dat,
    on=['lon', 'lat', 'day'],
    how='left'
)

rows_after = len(all_features)
print(f"After merge  : {rows_after:,} rows × {all_features.shape[1]} cols")

assert rows_before == rows_after, \
    f"Row count changed after left join: {rows_before:,} -> {rows_after:,}"
print("Row count unchanged — left join OK.")

Before merge : 127,478,960 rows × 17 cols
After merge  : 127,478,960 rows × 18 cols
Row count unchanged — left join OK.


## 7. Post-Merge Validation

Confirm the schema is correct (18 columns) and review missing rates.
Key expectations:
- `LAI` should be ~3% missing (cloud-contaminated retrievals)
- All other column missing rates should be unchanged from the base table

In [14]:
print(f"Shape   : {all_features.shape}")
print(f"Columns : {list(all_features.columns)}")

Shape   : (127478960, 18)
Columns : ['day', 'lat', 'lon', 'SWE', 'year', 'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_100hr', 'max_air_temperature', 'max_relative_humidity', 'min_air_temperature', 'min_relative_humidity', 'precipitation_amount', 'specific_humidity', 'surface_downwelling_shortwave_flux_in_air', 'wind_from_direction', 'wind_speed', 'population_density', 'LAI']


In [15]:
missing_rates = all_features.isnull().mean().mul(100).rename('missing_%')
print("Missing rate per column (%):\n")
print(missing_rates.to_string())

lai_missing = missing_rates['LAI']
print(f"\nLAI missing: {lai_missing:.4f}%")
if lai_missing > 10.0:
    print("  NOTE: >10% missing — check LAI grid / date alignment.")
else:
    print("  OK — within expected range.")

Missing rate per column (%):

day                                           0.000000
lat                                           0.000000
lon                                           0.000000
SWE                                           1.709074
year                                          0.000000
dead_fuel_moisture_1000hr                     0.167830
dead_fuel_moisture_100hr                      0.167830
max_air_temperature                           0.116906
max_relative_humidity                         0.167830
min_air_temperature                           0.116906
min_relative_humidity                         0.167832
precipitation_amount                         57.369750
specific_humidity                             0.167830
surface_downwelling_shortwave_flux_in_air     0.167830
wind_from_direction                           0.267880
wind_speed                                    0.167830
population_density                            0.076640
LAI                                

## 8. Save Output

Write the merged feature table to Parquet. This file is the input for
the next merge step (subregion labels and vegetation type features).

In [16]:
output_path = os.path.join(FEATURE_DATA_DIR, OUTPUT_FILE)

all_features.to_parquet(output_path, index=False)

print(f"Saved -> {output_path}")
print(f"File size  : {os.path.getsize(output_path) / 1e9:.2f} GB")
print(f"Final shape: {all_features.shape[0]:,} rows × {all_features.shape[1]} cols")
print(f"Columns    : {list(all_features.columns)}")

del all_features, lai_dat
gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Weather_Population_LAI_Merged.parquet
File size  : 2.42 GB
Final shape: 127,478,960 rows × 18 cols
Columns    : ['day', 'lat', 'lon', 'SWE', 'year', 'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_100hr', 'max_air_temperature', 'max_relative_humidity', 'min_air_temperature', 'min_relative_humidity', 'precipitation_amount', 'specific_humidity', 'surface_downwelling_shortwave_flux_in_air', 'wind_from_direction', 'wind_speed', 'population_density', 'LAI']


0

## 9. Summary

| Phase | Step | Description | Key Result |
|-------|------|-------------|------------|
| A | Load LAI | `LAI.parquet` from `Clean_Data/LAI_Data/` | 147M rows × 4 cols |
| A | Date Range | Check LAI spans 1994–2021 | Confirmed |
| A | Key Check | Assert unique `(time, lat, lon)` triplets | No duplicates |
| B | Rename | `time` → `day` in LAI before join | Column aligned |
| B | Load Base | `Weather_Population_Merged.parquet` | 127M rows × 17 cols |
| B | Date Coverage | Verify weather range ⊆ LAI range | 2020-09-30 ≤ 2021-12-27 |
| C | Left Join | Merge on `(lon, lat, day)` | Row count unchanged |
| C | Validate | Shape (18 cols) + missing rates | LAI ~3% NaN (expected) |
| C | Save | `Weather_Population_LAI_Merged.parquet` | 127M rows × 18 cols |